# Duplicate-screen sensitivity and calibration checks
Attach the frozen supplemental inputs and the MSLD v2.0 release. Run on CPU.


In [ ]:
import importlib.metadata as metadata
import subprocess, sys
required = {'numpy':'2.1.3','pandas':'2.3.3','scipy':'1.16.3','scikit-learn':'1.6.1','timm':'1.0.29','Pillow':'13.0.0','huggingface_hub':'1.33.0','safetensors':'0.8.0','matplotlib':'3.10.8'}
changes=[]
for name,version in required.items():
    try: installed=metadata.version(name)
    except metadata.PackageNotFoundError: installed=None
    if installed!=version: changes.append(f'{name}=={version}')
if changes: subprocess.run([sys.executable,'-m','pip','install','--quiet','--no-deps',*changes],check=True)
print('Pinned analysis packages ready; Kaggle torch and torchvision retained.')
subprocess.run([sys.executable,'-m','pip','install','--quiet','--no-deps','opencv-python-headless==4.13.0.92'],check=True)


In [ ]:
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
import hashlib,json,shutil,stat,sys
EXPECTED_SHA = "8d6fe11c06d5ce9af25f227d99fd63fdc4854dba924e01a584b0126efad4c5f5"
def file_sha(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(2**20),b''):h.update(b)
    return h.hexdigest()
search=Path('/kaggle/input');markers=list(search.rglob('supplemental_manifest.json'))
if not any(file_sha(p)==EXPECTED_SHA for p in markers):
    for archive in search.rglob('MSLD_Supplemental_Inputs.zip'):
        destination=Path('/tmp/msld_supplemental_bootstrap')/file_sha(archive)
        with ZipFile(archive) as z:
            if sum(i.file_size for i in z.infolist())>100*2**20:raise ValueError('Expected the small supplemental input ZIP.')
            for info in z.infolist():
                rel=PurePosixPath(info.filename.replace('\\','/'));target=destination.joinpath(*rel.parts).resolve()
                if rel.is_absolute() or '..' in rel.parts or any(':' in a for a in rel.parts) or stat.S_ISLNK(info.external_attr>>16) or not target.is_relative_to(destination.resolve()):raise ValueError('Unsafe ZIP path')
                if info.is_dir():target.mkdir(parents=True,exist_ok=True)
                else:
                    target.parent.mkdir(parents=True,exist_ok=True)
                    with z.open(info) as src,target.open('wb') as dst:shutil.copyfileobj(src,dst)
        markers+=list(destination.rglob('supplemental_manifest.json'))
matches=[p for p in markers if file_sha(p)==EXPECTED_SHA]
if not matches:raise FileNotFoundError('Attach MSLD_Supplemental_Inputs.zip as a private Kaggle dataset.')
INPUTS=sorted(matches)[0].parent
index=json.loads((INPUTS/'supplemental_manifest.json').read_text())
for rel,rec in index['files'].items():
    p=(INPUTS/rel).resolve()
    if not p.is_relative_to(INPUTS.resolve()) or not p.is_file() or p.stat().st_size!=rec['bytes'] or file_sha(p)!=rec['sha256']:raise ValueError('Missing or changed input: '+rel)
CODE=Path('/kaggle/working/supplemental_code');CODE.mkdir(exist_ok=True)
for source in (INPUTS/'code').glob('*.py'):shutil.copyfile(source,CODE/source.name)
print('Supplemental package verified. Analysis plan:',file_sha(INPUTS/'analysis_plan.json'))


In [ ]:
base=[sys.executable,str(CODE/'supplemental_audit.py'),'--inputs',str(INPUTS),'--expected-sha',EXPECTED_SHA]


In [ ]:
subprocess.run(base,check=True)
